# liar_grid

> Phase 15 grids and the frozen analysis plan: `traceGrid` (Phase 14's Part B `baseline` cells, replayed from cache), `forecastCells` (the numpy forecast's cells, optionally with a rule-agent variant or another liar share), `liarShareGrid` (Part D, an eighth of the agents liars), `liarForecast` (the Part D forecast), `TRACE_PLAN`, and the changes with team size: `nChange` (the Phase 14 forecast rule) and `numpyChanges`.

In [ ]:
#| default_exp liar_grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

from explRL.clue_grid import CLUE_SETTINGS, _board, clueCell
from explRL.fix_grid import fixCell
from explRL.liar_trace import VARIANTS, traceCells
from explRL.llm_grid import HAIKU
from explRL.llm_report import NAN, _f
from explRL.scale_grid import SCALE_SEEDS, scaleGrid

FORECAST_N, FORECAST_SEEDS = (4, 8, 16, 32, 64), range(2000, 2020)
D_N = (8, 32)
TRACE_PLAN = dict(
    version=1, frozen=True, model=HAIKU, settings=CLUE_SETTINGS, turns=40, channel='blackboard', arm='baseline',
    cells=dict(haiku="Phase 14 study scaleB, level baseline and baseline|liar (N // 4 liars), N 8, 16, 32, seeds 900-904; "
                     'replayed with the cache-only client (a miss raises; nothing is spent)',
               numpy='rule agents as tools/scale_forecast.py: N 4, 8, 16, 32, 64, clean and N // 4 liars, seeds 2000-2019'),
    measures=dict(
        M1='falseVisible: open subtasks showing a false post / open subtasks, mean over honest turns; contaminated: honest '
           "submissions whose known clues (known() at the turn's start) for the subtask include a false residue / honest submissions",
        M2='wrongContam, wrongClean: wrong results / submissions, contaminated and uncontaminated honest submissions',
        M3="suppressed: slots (subtask, clue) first posted by a liar whose true residue no honest agent posted later / such slots",
        M4='relayPerLie: false relays (an honest share of a false residue a liar posted earlier for that slot) / liar posts; '
           'relayAny: liar posts relayed at least once / liar posts; honestErrors: other false honest shares / honest shares',
        M5='liarHeld: subtasks with a clue held by a liar / subtasks created; solvedLiarHeld, solvedOther: solve rates',
        M6='recoverRounds: rounds from an honest wrong answer to the solve (solved subtasks); contamUnsolved: subtasks with a '
           'contaminated submission never solved / subtasks with one'),
    accounting=dict(
        terms='a: contaminated penalties (liar cell); b: other penalties (liar cell) - clean penalties; c, d, e, f: value '
              'honest agents solved in the clean cell and no honest agent in the liar cell, assigned e (a liar solved it), '
              'c (a false post on it), d (a liar-held clue), f (the rest), each / (N x turns); g: - value honest agents '
              'solved only in the liar cell / ((N - N // 4) x turns); h: value solved by honest agents in both x '
              '(1 / (N x turns) - 1 / ((N - N // 4) x turns))',
        check='a + ... + h = welfareHonest clean - with liars, to 1e-12'),
    variants=dict(relay='after its own clues, a clue known from the board (not held) for a subtask a teammate has open, '
                        'not yet posted by this agent',
                  first='the earliest sighting of a clue wins (own clues still win)',
                  repost='its own slot counts as shared only when a post of it carries the true residue'),
    change='mean at N 32 - mean at N 8, 95% bootstrap interval resampling seeds within each N (unpaired), 2000 draws, '
           'seed 0; sign + / - when the interval excludes 0, else 0 (the Phase 14 forecast rule)',
    hypotheses=dict(
        H15_1="contaminated: Haiku's change sign + or 0; numpy's sign -",
        H15_2='Haiku relayPerLie above 0 at every N (mean), its change sign + or 0',
        H15_3="Haiku's change in (a + c) / its change in the total above 0.5 (bootstrap CI reported)",
        H15_4="some variant's honest-loss change sign equals Haiku's (+); retrodiction",
        H15_5='Part D (optional): the chosen variant forecast sign of the honest-loss change N 8 -> 32 at nLiar = N // 8 '
              'equals Haiku\'s'),
    partD=dict(N=list(D_N), nLiar='N // 8', seeds=[900, 904], clean="Phase 14's scaleB baseline rows",
               choose="among variants passing H15-4, the one whose honest-loss change is closest to Haiku's; none: no Part D"),
    secondary='N 16 and 32 pooled against N 8', nboot=2000, ci=0.95, exploratory=True)


def traceGrid():
    "Phase 14's scaleB baseline cells, clean and with N // 4 liars (30 cells, the same keys)."
    return [c for c in scaleGrid() if c['level'] in ('baseline', 'baseline|liar')]


def forecastCells(variant=None, share=4, sizes=FORECAST_N, seeds=FORECAST_SEEDS, s=CLUE_SETTINGS):
    """The numpy forecast's cells (tools/scale_forecast.py: study scaleNumpy, levels clean and liar, N // share liars);
    with a variant, cfg.world['variant'] is set and the study is liarVariant."""
    if variant not in VARIANTS:
        raise ValueError(f'unknown variant {variant}')
    w = dict(_board(s), guess=s['guess'], share='needed', **({'variant': variant} if variant else {}))
    return [clueCell('liarVariant' if variant else 'scaleNumpy', kind, N, sd, 'numpy', s=s, channel='blackboard', turns=40,
                     nLiar=N // share if kind == 'liar' else 0, world=w)
            for N in sizes for kind in ('clean', 'liar') for sd in seeds]


def liarShareGrid(seeds=SCALE_SEEDS):
    "Part D: study liarD, baseline, N 8 and 32 with N // 8 liars (the clean rows are Phase 14's)."
    return [fixCell('liarD', 'baseline', N, sd, N // 8) for N in D_N for sd in seeds]


def nChange(lo, hi, n=2000, seed=0):
    "The Phase 14 forecast rule (tools/scale_forecast.py change): mean(hi) - mean(lo), bootstrap within each side, sign."
    lo, hi = np.asarray([x for x in lo if x == x], float), np.asarray([x for x in hi if x == x], float)
    if not len(lo) or not len(hi):
        return dict(mean=NAN, ci=[NAN, NAN], sign='0')
    rng = np.random.default_rng(seed)
    d = rng.choice(hi, (n, len(hi))).mean(1) - rng.choice(lo, (n, len(lo))).mean(1)
    ci = [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))]
    return dict(mean=float(hi.mean() - lo.mean()), ci=ci, sign='+' if ci[0] > 0 else '-' if ci[1] < 0 else '0')


def _boot(f, sides, n=2000, seed=0):
    "Point value and 95% interval of f(*arrays) resampling each array's rows (seeds) independently."
    rng, sides = np.random.default_rng(seed), [np.asarray(x, float) for x in sides]
    v = np.array([f(*(x[rng.integers(0, len(x), len(x))] for x in sides)) for _ in range(n)])
    v = v[np.isfinite(v)]
    return dict(mean=float(f(*sides)), ci=[float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] if len(v) else [NAN, NAN])


def _pairs(runs):
    "{N: {seed: {'clean': run, 'liar': run}}} from {key: TraceRun} (liar when nLiar > 0)."
    out = {}
    for tr in runs.values():
        r = tr.rows[0]
        out.setdefault(int(_f(r['N'])), {}).setdefault(int(_f(r['seed'])), {})['liar' if tr.nLiar else 'clean'] = tr
    return out


def numpyChanges(rows, metrics=('pooledShare', 'honestLoss')):
    "Per metric: the curve over N (means) and the change N 8 -> 32, from rows of clean and liar cells (pairs by N, seed)."
    by = {}
    for r in rows:
        by.setdefault(int(_f(r['N'])), {}).setdefault(int(_f(r['seed'])), {})['liar' if _f(r.get('nLiar', 0)) > 0 else 'clean'] = r
    val = {N: dict(pooledShare=[d['clean']['pooledShare'] for d in s.values() if 'clean' in d],
                   honestLoss=[d['clean']['welfareHonest'] - d['liar']['welfareHonest'] for d in s.values() if len(d) == 2])
           for N, s in by.items()}
    return {m: dict(curve={N: float(np.mean(v[m])) if v[m] else NAN for N, v in sorted(val.items())},
                    change=nChange(val.get(8, {}).get(m, []), val.get(32, {}).get(m, []))) for m in metrics}


def liarForecast(variant, seeds=FORECAST_SEEDS, workers=1):
    """Part D's forecast: the honest-loss change N 8 -> 32 with N // 8 liars, for the variant and the unchanged rule agent."""
    out = {}
    for v in dict.fromkeys((variant, None)):
        runs = traceCells(forecastCells(v, 8, D_N, seeds), workers=workers)
        out[v or 'rule'] = numpyChanges([tr.rows[0] for tr in runs.values()], ('honestLoss',))['honestLoss']
    return out

## Tests

In [ ]:
import json
from explRL.liar_grid import *
from explRL.bench_grid import rowKey
from explRL.coop_grid import coopHash
from explRL.fix_grid import requestKey
from explRL.llm_runs import LLMConfig, cellRow
from explRL.scale_grid import scaleGrid
from explRL.liar_trace import VARIANTS

key = lambda c: rowKey(cellRow(c, LLMConfig.fromDict(c['cfg']), dict(episode=0)))
g = traceGrid()
assert len(g) == 30 and {c['level'] for c in g} == {'baseline', 'baseline|liar'}
assert {(c['cfg']['N'], c['cfg'].get('nLiar', 0)) for c in g} == {(8, 0), (16, 0), (32, 0), (8, 2), (16, 4), (32, 8)}
assert {key(c) for c in g} <= {key(c) for c in scaleGrid()}          # Phase 14's keys, so the cache holds every reply
f = forecastCells()
assert len(f) == 5 * 2 * 20 and len({key(c) for c in f}) == 200
assert all(c['cfg'].get('nLiar', 0) == (c['cfg']['N'] // 4 if c['level'] == 'liar' else 0) for c in f)
# the forecast's cells exactly as tools/scale_forecast.py builds them
from explRL.clue_grid import CLUE_SETTINGS, _board, clueCell
s = CLUE_SETTINGS
tool = [clueCell('scaleNumpy', kind, N, sd, 'numpy', s=s, channel='blackboard', turns=40, nLiar=nL,
                 world=dict(_board(s), guess=s['guess'], share='needed'))
        for N in (4, 8, 16, 32, 64) for kind, nL in (('clean', 0), ('liar', N // 4)) for sd in range(2000, 2020)]
assert sorted(json.dumps(c, sort_keys=True) for c in f) == sorted(json.dumps(c, sort_keys=True) for c in tool)
v = forecastCells('relay')
assert all(c['cfg']['world']['variant'] == 'relay' and c['study'] == 'liarVariant' for c in v)
assert {requestKey(c) for c in v}.isdisjoint({requestKey(c) for c in f})
d = liarShareGrid()
assert len(d) == 10 and {(c['cfg']['N'], c['cfg']['nLiar']) for c in d} == {(8, 1), (32, 4)} and all(c['study'] == 'liarD' for c in d)
assert set(TRACE_PLAN['hypotheses']) == {f'H15_{i}' for i in range(1, 6)} and set(TRACE_PLAN['measures']) == {f'M{i}' for i in range(1, 7)}
assert set(TRACE_PLAN['variants']) == set(VARIANTS) - {None}
print('TRACE_PLAN', coopHash(TRACE_PLAN), 'frozen' if TRACE_PLAN['frozen'] else 'not frozen')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()